In [1]:
#!pip install azure-search-documents
#!pip install azure-identity
#!pip install openai
#!pip install --upgrade typing_extensions
#dbutils.library.restartPython()

In [2]:
# 認証とindexの定義
index_name="20241222_all_index-rag-extractive-summary-verify"

#Azure AI Searchの情報を設定
aisearch_service_name = 'pdfsearcher'
aisearch_api_version = "2024-05-01-preview"
aisearch_api_key = "FhhwPvk7exA2g7NaMhpHWRzvYMtDPeMdFMzHVM0BQNAzSeD1W6Jw"
aisearch_endpoint = "https://pdfsearcher.search.windows.net"

openai_api_key = "ASecZYTdvbYZKKwGNiitNIARu7AKQd05G9bYhdLjpDYbwSoV3NdPJQQJ99ALACYeBjFXJ3w3AAABACOG0pBY"
openai_endpoint = "https://eastustago.openai.azure.com/"
openai_api_version = "2024-10-01-preview"


excel_path = './回答結果.xlsx'
result_file = "extract_summary_results"

qa_file_path = './質問.xlsx'
sheet_name = "20240909"

In [3]:
# 質問からキーワードを抽出
def exractKeyword(input,openai_endpoint,openai_api_key,openai_api_version):
    import openai
    client = openai.AzureOpenAI(
        azure_endpoint=openai_endpoint,
        api_key=openai_api_key,
        api_version=openai_api_version
    )

    # 質問に対する回答を生成する関数
    def get_answer(question):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages= [{"role": "user", "content": question}]
        )
        return response.choices[0].message.content.strip()


    # 質問を入力して回答を取得
    question = f"""以下の質問から重要なキーワードを３つ抽出してください。そのキーワードはAIサーチの検索に使用されます。応答はプログラムで使いやすいようにJsonにしてください\n###質問\n{input}
    """
    answer = get_answer(question)
    # print("質問:", question)
    print("回答:", answer)
    # 抽出したキーワードを配列に
    import json
    ans1 = answer.replace("```","")
    ans2= ans1.replace("json","")
    # print(ans2)
    data = json.loads(ans2.replace("```",""))
    # キーワードの配列を取得
    keywords_array = data["keywords"]
    # 配列を表示
    # print(keywords_array)
    return keywords_array

In [4]:
# キーワードからフルテキスト検索
def fullTextSearch(keywords_array):
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents.indexes import SearchIndexClient
    from azure.search.documents import SearchClient
    import json

    # Set Credentials
    credential = AzureKeyCredential(aisearch_api_key)

    # Run an empty query (returns selected fields, all documents)
    search_client = SearchClient(endpoint=aisearch_endpoint,
                        index_name=index_name,
                        credential=credential)
    rag_list = list()
    for keywd in keywords_array:
        results =  search_client.search(query_type='simple',
            search_text=keywd,
            # select='Title,MajorCategory,SubCategory,Content,Summary,TableInfo,TableDesc,GraphInfo,GraphDesc',
            # select を指定しない場合は、スキーマで取得可能なフィールドはすべて取得
            top=3,#全件だと多くなるのでとりあえず3件に設定
            include_total_count=True)

        print (f'Total Documents Matching Query {keywd}:\n', results.get_count())
        for result in results:
            print(f'score:{result["@search.score"]} title:{result["Title"]}')
            text = json.dumps(result,ensure_ascii=False)
            rag_list.append(text)
    return rag_list

In [5]:
# 質問を検索に適した形にする
def analyzeQuery(query):
    import openai
    client = openai.AzureOpenAI(
        azure_endpoint=openai_endpoint,
        api_key=openai_api_key,
        api_version="2024-04-01-preview"
    )

    # 質問に対する回答を生成する関数
    def get_answer(question):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages= [{"role": "user", "content": question}]
        )
        return response.choices[0].message.content.strip()


    # 質問を入力して回答を取得
    question = f"""以下の質問をAIサーチでハイブリッド検索するのに適した形にしてください。応答はAIサーチのハイブリッド検索に使用されます。応答はその後のプログラムで使いやすいようにJsonにしてください\n###質問\n{query}
    """
    answer = get_answer(question)
    # print("質問:", question)
    # print("回答:", answer)
    import json
    tmp1 = answer.replace("```","")
    new_answer= tmp1.replace("json","")
    print(f'再構築した質問:{new_answer}')
    return new_answer


In [6]:
# 最適化した質問文でハイブリッド検索
def hybridSearch(query):
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient
    from azure.search.documents.indexes import SearchIndexClient
    from azure.search.documents.models import VectorizedQuery


    def get_embeddings(text: list[str]):
        # There are a few ways to get embeddings. This is just one example.
        import openai
        # print(f"search_endpoint: {search_endpoint} api_key: {search_api_key} api_version: {api_version}")
        client = openai.AzureOpenAI(
            azure_endpoint=openai_endpoint,
            api_key=openai_api_key,
            api_version="2024-06-01"
        )
        embedding = client.embeddings.create(input=[text], model="text-embedding-3-large")
        # print(f"embedding.data={embedding.data[0].embedding}")
        return embedding.data[0].embedding

    embd = get_embeddings(query)
    # print(f"resp={resp}")
    vector_query2 = VectorizedQuery(vector=embd, k_nearest_neighbors=3, fields="ContentVector")
    vector_query3 = VectorizedQuery(vector=embd, k_nearest_neighbors=3, fields="SummaryVector")

    credential = AzureKeyCredential(aisearch_api_key)
    search_client = SearchClient(endpoint=aisearch_endpoint,
                        index_name=index_name,
                        credential=credential)
    results =  search_client.search(query_type='semantic',
        search_text=query,
        vector_queries=[vector_query2,vector_query3],
        # selectは不要
        semantic_configuration_name="q2-semantic_configuration",
        # include_total_count=True # 近似値だしパフォーマンス影響あるらしいので一旦削除
        top=3
        )
    import json
    rag_list = list()
    for result in results:
        print(f'hybrid score:{result["@search.score"]} title:{result["Title"]}')
        text = json.dumps(result,ensure_ascii=False)
        rag_list.append(text)
    return rag_list

In [7]:
# 回答生成関数

import openai
client = openai.AzureOpenAI(
    azure_endpoint=openai_endpoint,
    api_key=openai_api_key,
    api_version="2024-04-01-preview"
)

# 回答生成用のプロンプト
system_prompt = """あなたは、20年の経験があるプロフェッショナルな塾講師です。分かり易い説明に定評があります。
生徒からの質問に対して事実に基づいた見地から、回答やアドバイスをしてください。
"""

# 質問に対する回答を生成する関数
def get_answer(question, system_prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ],
    )
    return response.choices[0].message.content.strip()

In [8]:
# promptを作成する関数
def createPrompt(query, rag_list):
    prompt = f"""以下の質問文に対し、データソースを基にして適切に回答してください。
    数式や化学式はLaTeX表記のまま出力せず、人が読みやすい形式に直して出力してください。\n
    例：2 ext{{NaHCO}}_3 (s) ightarrow ext{{Na}}_2ext{{CO}}_3 (s) + ext{{CO}}_2 (g) + ext{{H}}_2ext{{O}} (g) ⇒ \n
    2NaHCO₃ (固体) → Na₂CO₃ (固体) + CO₂ (気体) + H₂O (気体)
    \n\n### 質問文\n
    {query}
    \n\n### データソース\n
    """
    import json
    for src in rag_list:
        item = json.loads(src)
        ragitem = f"""Title:{item['Title']}\nMajorCategory:{item['MajorCategory']}\nSubCategory:{item['SubCategory']}\nContent:{item['Content']}\n
    ---\n"""
        prompt += ragitem
    return prompt

# 回答生成

# 回答生成2（RateLimitError対策版）

In [9]:
import json

query = "負の加減算の計算方法を教えてください"
print(f"process start:{query}")
#human_answer = row['回答']
# keywords = exractKeyword(query, openai_endpoint, openai_api_key, openai_api_version)
# rag_list = fullTextSearch(keywords)
# new_query = analyzeQuery(query)
rag_list = hybridSearch(query)
#rag_list.extend(rag_list2)
prompt = createPrompt(query, rag_list)
documenttitle = ""
for src in rag_list:
    item = json.loads(src)
    ragitem = f"""Title:{item['Title']} MajorCategory:{item['MajorCategory']} SubCategory:{item['SubCategory']}\n"""
    documenttitle += ragitem

# 回答生成
rag_answer = get_answer(prompt, system_prompt)

qa_results = []
qa_results.append({
    #'質問ナンバー' : index + 1,
    '質問' : query,
    'LLM回答' : rag_answer,
    '検索されたドキュメント':documenttitle
})

print(qa_results)

process start:負の加減算の計算方法を教えてください
hybrid score:0.05000000447034836 title:["{'id': 5, 'file_name': '1_08-39_1章_元.pdf', 'file_path': 'C:\\\\Users\\\\banqu\\\\AppData\\\\Local\\\\Temp\\\\tmpmscsuezn.pdf', 'page_number': 1, 'type': '章タイトル', 'figure_table_type': None, 'content': '【文字と式】', 'region': [0.7877, 1.1632, 1.4619, 1.1632, 1.4619, 1.3109, 0.7877, 1.3109]}", "{'id': 8, 'file_name': '1_08-39_1章_元.pdf', 'file_path': 'C:\\\\Users\\\\banqu\\\\AppData\\\\Local\\\\Temp\\\\tmpmscsuezn.pdf', 'page_number': 1, 'type': '章タイトル', 'figure_table_type': None, 'content': '【文字にあてはまる数2】', 'region': [3.4703, 2.6539, 4.8498, 2.6539, 4.8498, 2.809, 3.4703, 2.809]}", "{'id': 9, 'file_name': '1_08-39_1章_元.pdf', 'file_path': 'C:\\\\Users\\\\banqu\\\\AppData\\\\Local\\\\Temp\\\\tmpmscsuezn.pdf', 'page_number': 1, 'type': '章タイトル', 'figure_table_type': None, 'content': '【文字にあてはまる数1」', 'region': [0.3159, 2.6561, 1.6417, 2.6561, 1.6417, 2.8094, 0.3159, 2.8094]}", "{'id': 19, 'file_name': '1_08-39_1章_元.pdf', 'file

In [10]:
len(qa_results)

1

In [11]:
import re

# 検索結果からページ番号とファイル名の組みを抽出する関数
def search_page_and_file(text):
    # 正規表現パターンを作成
    page_pattern = r"'page_number': (\d+)"
    file_pattern = r"'file_name': '([^']+)'"
    # 正規表現で全ての数字を抽出
    page_matches = re.findall(page_pattern , text)
    file_matches = re.findall(file_pattern , text)
    page_numbers = [int(match) for match in page_matches]
    file_names = [match for match in file_matches]

    # 要素の組み合わせをタプルとして作成し、一意の組み合わせを抽出
    unique_combinations = set(zip(page_numbers, file_names))

    # unique_combinationsをソートするための関数
    def sort_key(item):
        file_name, page_number = item
        return file_name, page_number
    # unique_combinationsをソート
    sorted_combinations = sorted(unique_combinations, key=sort_key)

    return sorted_combinations

In [12]:
rag_list

['{"SubCategory": "default_subcategory", "TableDesc": "[\\"{\'id\': 46, \'content\': \'ファイル名：1_08-39_1章_元\\\\\\\\n\\\\\\\\n表名：\\\\\\\\u3000チーム成績表\\\\\\\\n\\\\\\\\n表1の内容の説明：\\\\\\\\u3000この表は、4つのサッカーチームの試合結果を示しており、各チームの順位、試合数、勝利数、引き分け数、敗北数、得点、失点、得失点差、勝ち点が記載されています。\'}\\", \\"{\'id\': 49, \'content\': \'ファイル名：1_08-39_1章_元\\\\\\\\n\\\\\\\\n表名：\\\\\\\\u3000表#2\\\\\\\\n\\\\\\\\n表1の内容の説明：\\\\\\\\u3000この表は、特定の陸上競技選手の100メートル走のタイム、風速、名前、所属国、日付を示しています。選手ごとの記録が整理されており、風速の影響を受けたタイムが記載されています。\'}\\", \\"{\'id\': 107, \'content\': \'ファイル名：1_08-39_1章_元\\\\\\\\n\\\\\\\\n表名：\\\\\\\\u3000表#3\\\\\\\\n\\\\\\\\n表1の内容の説明：\\\\\\\\u3000この表は、5回のテストにおける点数を示しており、各回の得点が記録されています。\'}\\", \\"{\'id\': 154, \'content\': \'ファイル名：1_08-39_1章_元\\\\\\\\n\\\\\\\\n表名：\\\\\\\\u3000表#4\\\\\\\\n\\\\\\\\n表1の内容の説明：\\\\\\\\u3000この表は、特定の数値範囲におけるデータの分布を示しており、各列は異なるカテゴリ（a, b, c, d, e）に関連する数値を表しています。\'}\\", \\"{\'id\': 286, \'content\': \'ファイル名：1_08-39_1章_元\\\\\\\\n\\\\\\\\n表名：\\\\\\\\u3000表#5\\\\\\\\n\\\\\\\\n表1の内容の説明：\\\\\\\\u3000この表は、

In [13]:
search_page_and_file(rag_list[0])[0][1]

'1_08-39_1章_元.pdf'

In [15]:
# リターンする項目のリスト
response_results = []

In [16]:
print("# 質問：", "\n",qa_results[0]['質問'],"\n")
# RAG回答に含まれる"/"の削除
pattern = r"\\."
cleaned_text = re.sub(pattern, "", qa_results[0]['LLM回答'])
cleaned_text = cleaned_text.replace("\n\n", "\n")
print("# 回答：", "\n",cleaned_text,"\n")
print("# 参照ファイル (関連性の高い順)：")
# 参照ファイルのリストを作成
reference_text_list = []
# ファイルの有無を確認するためのフラグ
file_exists = False
# ファイルが無いことを確認するためのフラグ
file_not_exists= False
for rag_text in rag_list:
    try:
        print(" ", search_page_and_file(rag_text)[0][1])
        reference_text_list.append(search_page_and_file(rag_text)[0][1])
        file_exists = True
    except IndexError:
        if file_exists:
            pass
        else:
            if not file_not_exists:
                print(" 該当する資料がありません。")
                reference_text_list.append("該当する資料がありません。")
                file_not_exists = True


response_results.append({
        '質問' : qa_results[0]['質問'],
        'LLM回答' : cleaned_text,
        '検索されたドキュメント':reference_text_list
    })

print(response_results)


# 質問： 
 負の加減算の計算方法を教えてください 

# 回答： 
 負の加減算の計算方法について説明します。
### 負の数の加減算の基本ルール
1. **同符号の数の加算**:
   - 同じ符号の数（正の数同士または負の数同士）を足すと、符号はそのままで、絶対値を足します。
   - 例: 3 + 5 = 8（正の数同士）
   - 例: -3 + (-5) = -8（負の数同士）
2. **異符号の数の加算**:
   - 異なる符号の数を足す場合は、絶対値の大きい方から小さい方を引き、結果の符号は絶対値が大きい方の符号を取ります。
   - 例: 5 + (-3) = 5 - 3 = 2（正の数と負の数）
   - 例: -5 + 3 = -5 + (-3) = -2（負の数と正の数）
3. **引き算の計算**:
   - 引き算は、加算に変換できます。引き算の数の符号を変えて加算します。
   - 例: 5 - 3 = 5 + (-3) = 2
   - 例: -5 - 3 = -5 + (-3) = -8
### 具体例
- **例1**: 7 - 10
  - これは 7 + (-10) と考えます。
  - 絶対値の大きい方は 10 なので、10 - 7 = 3 となり、結果は -3 です。
  - よって、7 - 10 = -3。
- **例2**: -4 + 6
  - 異符号の加算です。
  - 絶対値の大きい方は 6 で、6 - 4 = 2 となります。
  - 結果は正の数なので、-4 + 6 = 2。
- **例3**: -8 - 5
  - これは -8 + (-5) と考えます。
  - 同符号の加算なので、-8 - 5 = -13 です。
### まとめ
負の数の加減算は、同符号の数はそのまま足し、異符号の数は絶対値を引いて大きい方の符号を取ります。また、引き算は加算に変換して計算します。これらのルールを使って、負の数の計算を正確に行うことができます。 

# 参照ファイル (関連性の高い順)：
  1_08-39_1章_元.pdf
  1年_中学校理科_指導書_板書例_p56.pdf
  1年_中学校理科_指導書_板書例_p99.pdf
[{'質問': '負の加減算の計算方法を教えてください', 'LLM回答': '負の加

In [14]:
# import re

# # サンプルの文字列
# text = "This is a sample string with \\a backslash and some \\b characters."

# # 正規表現パターンを作成して、\とその次の1文字を削除
# pattern = r"\\."
# cleaned_text = re.sub(pattern, "", text)

# print(f"削除後の文字列: {cleaned_text}")

削除後の文字列: This is a sample string with  backslash and some  characters.


# 回答生成を連続で実施し、データベースに保存する

In [9]:
# import pandas as pd
# import json
# qa_df = pd.read_excel(qa_file_path,sheet_name=sheet_name)
# display(qa_df)

In [96]:
rag_list

['{"ID": "1", "Content": "文字を使うと,いろいろな数や量を,1つの式で表せたね。\\n数や量を表す時に,や0などの記号の他に,aやxのような文字を使うことがある。\\nxにあてはまる数を求める時,x+8=21のように足し算の式になる場合、その逆の引き算でxを求めることができる。\\nxにあてはまる数を求める時,5×x=18のように掛け算の式になる場合、その逆の割り算でxを求めることができる。\\n5×x=18x=18÷5x=x\\nx+8=21x=21-8x=13\\n185\\n足し算の逆は引き算,掛け算の逆は割り算だったね。\\n文字にあてはまる数は,いろいろな数を文字に入れて探したね。\\n両方に同じ数をかけたり同じ数でわったりしても,比は変わらないことを学んだね。\\nある物の量を2とした時、別の物の量が3であることを,2:3と表す。このような割合の表し方を,比という。\\n割合と倍は、同じような使い方ができたね。\\n比がa:bで表される時,aをbでわった商を比の値という。比の値は,aがbの何倍にあたるかを表している。\\n1章正の数·負の数2章文字式\\n3章1次方程式\\n1正の数·負の数\\n章\\nチャプター1\\n20-5だから,那覇市の明日の最高気温は15°cだね。\\n札幌市の明日の最高気温は,-5-3ということかな。計算できないよ。\\n1節正の数·負の数\\n2節加法·減法\\n3節乗法·除法\\n4節数の集合\\n1正の数·負の数\\n2022年サッカーワールドカップグループe順位表\\nけつ\\n男子100m競走歴代10傑(2023年3月現在)\\n右の2つの温度計は、ある日の午前6時における新潟と鹿児島の気温を示しています。其々何°cを示しているでしょうか。また、「ー」のついた気温はどのようなことを表しているか考えてみましょう。\\nqquestion\\n5\\n0°cより低い気温に「ー」がついているね。\\n新潟\\n鹿児島\\n見方·考え方\\n0より小さいということかな。\\nどこに着目して考えればいいかな。\\n活動\\n目標▶「ー」のついた数の意味を考えよう。\\n0°cより2°c低い温度は,ーを使って-2°cと表し,\\n「マイナス2°c」と読む。これに対して,0°cより8°c高い温度は,+を

In [98]:
rag_list[0]

'{"ID": "1", "Content": "文字を使うと,いろいろな数や量を,1つの式で表せたね。\\n数や量を表す時に,や0などの記号の他に,aやxのような文字を使うことがある。\\nxにあてはまる数を求める時,x+8=21のように足し算の式になる場合、その逆の引き算でxを求めることができる。\\nxにあてはまる数を求める時,5×x=18のように掛け算の式になる場合、その逆の割り算でxを求めることができる。\\n5×x=18x=18÷5x=x\\nx+8=21x=21-8x=13\\n185\\n足し算の逆は引き算,掛け算の逆は割り算だったね。\\n文字にあてはまる数は,いろいろな数を文字に入れて探したね。\\n両方に同じ数をかけたり同じ数でわったりしても,比は変わらないことを学んだね。\\nある物の量を2とした時、別の物の量が3であることを,2:3と表す。このような割合の表し方を,比という。\\n割合と倍は、同じような使い方ができたね。\\n比がa:bで表される時,aをbでわった商を比の値という。比の値は,aがbの何倍にあたるかを表している。\\n1章正の数·負の数2章文字式\\n3章1次方程式\\n1正の数·負の数\\n章\\nチャプター1\\n20-5だから,那覇市の明日の最高気温は15°cだね。\\n札幌市の明日の最高気温は,-5-3ということかな。計算できないよ。\\n1節正の数·負の数\\n2節加法·減法\\n3節乗法·除法\\n4節数の集合\\n1正の数·負の数\\n2022年サッカーワールドカップグループe順位表\\nけつ\\n男子100m競走歴代10傑(2023年3月現在)\\n右の2つの温度計は、ある日の午前6時における新潟と鹿児島の気温を示しています。其々何°cを示しているでしょうか。また、「ー」のついた気温はどのようなことを表しているか考えてみましょう。\\nqquestion\\n5\\n0°cより低い気温に「ー」がついているね。\\n新潟\\n鹿児島\\n見方·考え方\\n0より小さいということかな。\\nどこに着目して考えればいいかな。\\n活動\\n目標▶「ー」のついた数の意味を考えよう。\\n0°cより2°c低い温度は,ーを使って-2°cと表し,\\n「マイナス2°c」と読む。これに対して,0°cより8°c高い温度は,+を使

In [125]:
import re

# 検索結果からページ番号とファイル名の組みを抽出する関数
def search_page_and_file(text):
    # 正規表現パターンを作成
    page_pattern = r"'page_number': (\d+)"
    file_pattern = r"'file_name': '([^']+)'"
    # 正規表現で全ての数字を抽出
    page_matches = re.findall(page_pattern , text)
    file_matches = re.findall(file_pattern , text)
    page_numbers = [int(match) for match in page_matches]
    file_names = [match for match in file_matches]

    # 要素の組み合わせをタプルとして作成し、一意の組み合わせを抽出
    unique_combinations = set(zip(page_numbers, file_names))

    # unique_combinationsをソートするための関数
    def sort_key(item):
        file_name, page_number = item
        return file_name, page_number
    # unique_combinationsをソート
    sorted_combinations = sorted(unique_combinations, key=sort_key)

    return sorted_combinations

In [122]:
# 抽出されたページ番号とファイル名のリスト
page_numbers = [1, 1, 1, 1, 1, 2, 2, 3, 3, 3, 5, 5, 7, 7, 7, 8, 9, 9, 10, 10, 10, 11, 11, 11, 11, 12, 12, 12, 12, 13, 13, 13, 14, 14, 15, 15, 15, 16, 16, 17, 18, 20, 21, 21, 22, 22, 23, 23, 25, 25, 25, 25, 26, 27, 27, 28, 29, 31, 31, 31, 31, 31, 32, 32]
file_names = ['0_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '2_08-39_1章_元.pdf', '00_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', 'xyz.pdf', 'abc.pdf']

# 要素の組み合わせをタプルとして作成し、一意の組み合わせを抽出
unique_combinations = set(zip(page_numbers, file_names))

# 結果を表示
print(f"一意の組み合わせ: {unique_combinations}")


一意の組み合わせ: {(12, '1_08-39_1章_元.pdf'), (23, '1_08-39_1章_元.pdf'), (3, '1_08-39_1章_元.pdf'), (14, '1_08-39_1章_元.pdf'), (5, '1_08-39_1章_元.pdf'), (27, '1_08-39_1章_元.pdf'), (25, '1_08-39_1章_元.pdf'), (16, '1_08-39_1章_元.pdf'), (1, '0_08-39_1章_元.pdf'), (7, '1_08-39_1章_元.pdf'), (18, '1_08-39_1章_元.pdf'), (29, '1_08-39_1章_元.pdf'), (20, '1_08-39_1章_元.pdf'), (31, '1_08-39_1章_元.pdf'), (22, '1_08-39_1章_元.pdf'), (1, '00_08-39_1章_元.pdf'), (9, '1_08-39_1章_元.pdf'), (11, '1_08-39_1章_元.pdf'), (2, '1_08-39_1章_元.pdf'), (13, '1_08-39_1章_元.pdf'), (32, 'xyz.pdf'), (1, '2_08-39_1章_元.pdf'), (15, '1_08-39_1章_元.pdf'), (26, '1_08-39_1章_元.pdf'), (17, '1_08-39_1章_元.pdf'), (28, '1_08-39_1章_元.pdf'), (8, '1_08-39_1章_元.pdf'), (10, '1_08-39_1章_元.pdf'), (1, '1_08-39_1章_元.pdf'), (32, 'abc.pdf'), (21, '1_08-39_1章_元.pdf')}


In [123]:
# unique_combinationsをソートするための関数
def sort_key(item):
    file_name, page_number = item
    return file_name, page_number

# unique_combinationsをソート
sorted_combinations = sorted(unique_combinations, key=sort_key)

# 結果を表示
print(f"ソートされた一意の組み合わせ: {sorted_combinations}")

ソートされた一意の組み合わせ: [(1, '00_08-39_1章_元.pdf'), (1, '0_08-39_1章_元.pdf'), (1, '1_08-39_1章_元.pdf'), (1, '2_08-39_1章_元.pdf'), (2, '1_08-39_1章_元.pdf'), (3, '1_08-39_1章_元.pdf'), (5, '1_08-39_1章_元.pdf'), (7, '1_08-39_1章_元.pdf'), (8, '1_08-39_1章_元.pdf'), (9, '1_08-39_1章_元.pdf'), (10, '1_08-39_1章_元.pdf'), (11, '1_08-39_1章_元.pdf'), (12, '1_08-39_1章_元.pdf'), (13, '1_08-39_1章_元.pdf'), (14, '1_08-39_1章_元.pdf'), (15, '1_08-39_1章_元.pdf'), (16, '1_08-39_1章_元.pdf'), (17, '1_08-39_1章_元.pdf'), (18, '1_08-39_1章_元.pdf'), (20, '1_08-39_1章_元.pdf'), (21, '1_08-39_1章_元.pdf'), (22, '1_08-39_1章_元.pdf'), (23, '1_08-39_1章_元.pdf'), (25, '1_08-39_1章_元.pdf'), (26, '1_08-39_1章_元.pdf'), (27, '1_08-39_1章_元.pdf'), (28, '1_08-39_1章_元.pdf'), (29, '1_08-39_1章_元.pdf'), (31, '1_08-39_1章_元.pdf'), (32, 'abc.pdf'), (32, 'xyz.pdf')]


In [124]:
import re

# サンプルの文字列
#text = "'file_name': '1_08-39_1章_元.pdf', 'file_name': '2_09-40_2章_次.pdf', 'file_name': '3_10-41_3章_後.pdf'"

# 正規表現パターンを作成
pattern = r"'file_name': '([^']+)'"

# 正規表現で全てのファイル名を抽出
matches = re.findall(pattern, text)
file_names = [match for match in matches]

print(f"抽出されたファイル名: {file_names}")

抽出されたファイル名: ['1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-39_1章_元.pdf', '1_08-3

In [103]:
len(file_names)

64

In [0]:
# 10件 11分
import pandas as pd
import json


qa_df = pd.read_excel(qa_file_path,sheet_name=sheet_name)

qa_results = []

for index, row in qa_df.iterrows():
    query = row['質問']
    print(f"process start:{query}")
    human_answer = row['回答']
    keywords = exractKeyword(query)
    rag_list = fullTextSearch(keywords)
    new_query = analyzeQuery(query)
    rag_list2 = hybridSearch(new_query)
    rag_list.extend(rag_list2)
    prompt = createPrompt(new_query, rag_list)
    documenttitle = ""
    for src in rag_list:
        item = json.loads(src)
        ragitem = f"""Title:{item['Title']} MajorCategory:{item['MajorCategory']} SubCategory:{item['SubCategory']}\n"""
        documenttitle += ragitem

    for i in range(1):
        rag_answer = get_answer(prompt)
        qa_results.append({
            '質問ナンバー' : index + 1,
            '質問' : query,
            'RAG回答' : rag_answer,
            '人間回答' : human_answer,
            '検索されたドキュメント':documenttitle
        })

qa_results_df = pd.DataFrame(qa_results)

display(qa_results_df)

process start:fチェーンの材質をSAE8620から18CrNiMo7-6に変更出来ないか検討しています。
SAE8620から18CrNiMo7-6への変更にあたっての心配点を洗い出す中でNiとCr添加の役割は下記と理解しています。
・Niは強度、焼入れ性、靭性を向上する。
・Cr : 焼入れ性、焼き戻し抵抗を向上＋浸炭促進効果がある。
故に、強度、耐摩耗性向上すると考えています。材料の一般的な知識がなく、上記の考え方が正しいのか、の確認をさせて頂きたいです。また、もし上記以外に材料成分が変わることによる背反、注意点があれば教えて頂けると幸いです。
回答: ```json
{
  "keywords": [
    "SAE8620",
    "18CrNiMo7-6",
    "材質変更"
  ]
}
```
Total Documents Matching Query SAE8620:
 0
Total Documents Matching Query 18CrNiMo7-6:
 260
score:8.602801 title:04-6_CZファスナーコート
score:8.164179 title:NAD116_応力計算の例6
score:7.858467 title:MKI00048_鋳鉄・鋳鋼の特性データ6
Total Documents Matching Query 材質変更:
 500
score:7.011875 title:材料リマインダ(T008)
score:6.539693 title:材料リマインダ(S026)
score:6.3615236 title:材料リマインダ(E111)
再構築した質問:
{
  "query": "チェーンの材質をSAE8620から18CrNiMo7-6に変更する際の心配点とNiおよびCr添加の役割について確認したい。具体的には、Niは強度、焼入れ性、靭性を向上させ、Crは焼入れ性、焼き戻し抵抗を向上させ、浸炭促進効果があると理解しているが、この考え方が正しいか。また、材料成分が変わることによる背反や注意点があれば教えてほしい。"
}

hybrid score:0.01666666753590107 title:材料リマインダ(T096)
hybrid score:0.01612903177

質問ナンバー,質問,RAG回答,人間回答,検索されたドキュメント
1,チェーンの材質をSAE8620から18CrNiMo7-6に変更出来ないか検討しています。 SAE8620から18CrNiMo7-6への変更にあたっての心配点を洗い出す中でNiとCr添加の役割は下記と理解しています。 ・Niは強度、焼入れ性、靭性を向上する。 ・Cr : 焼入れ性、焼き戻し抵抗を向上＋浸炭促進効果がある。 故に、強度、耐摩耗性向上すると考えています。材料の一般的な知識がなく、上記の考え方が正しいのか、の確認をさせて頂きたいです。また、もし上記以外に材料成分が変わることによる背反、注意点があれば教えて頂けると幸いです。,### 回答 チェーンの材質をSAE8620から18CrNiMo7-6に変更する際の心配点とNiおよびCr添加の役割について確認します。 #### NiおよびCr添加の役割 - **ニッケル (Ni)**: - **強度の向上**: Niは鋼の強度を向上させる効果があります。 - **焼入れ性の向上**: Niは焼入れ性を向上させ、鋼の硬化深さを増加させます。 - **靭性の向上**: Niは靭性を向上させ、衝撃に対する抵抗力を高めます。 - **クロム (Cr)**: - **焼入れ性の向上**: Crは焼入れ性を向上させ、鋼の硬化深さを増加させます。 - **焼き戻し抵抗の向上**: Crは焼き戻し抵抗を向上させ、高温での強度を維持します。 - **浸炭促進効果**: Crは浸炭を促進し、表面硬度を高めます。 これらの理解は正しいです。 #### 材料成分が変わることによる背反や注意点 1. **熱処理条件の変更**: - 18CrNiMo7-6はSAE8620よりも高い焼入れ性を持つため、熱処理条件を再評価する必要があります。特に、焼入れ温度や焼き戻し温度の調整が必要です。 2. **機械的特性の変化**: - 18CrNiMo7-6はNiとCrの添加により、強度と靭性が向上しますが、これにより材料の硬さが増加し、加工性が低下する可能性があります。 3. **摩耗特性の変化**: - NiとCrの添加により、摩耗抵抗が向上しますが、相手材との摩擦特性が変わる可能性があります。特に、チェーンのピンとブッシュの摩耗特性を再評価する必要があります。 4. **コストの増加**: - 18CrNiMo7-6は高価な合金元素を含むため、材料コストが増加する可能性があります。 5. **遅れ破壊のリスク**: - 高強度鋼は遅れ破壊のリスクが高まるため、特に高応力環境下での使用においては、遅れ破壊試験を実施し、適切な対策を講じる必要があります。 #### リマインダ - **材料リマインダ(T096)**: - チェーンの摩耗による伸びに関するリマインダがあります。ピンの表面処理やブッシュとの摺動で摩耗が発生しやすいことが指摘されています。クロマイジング処理後のポーラス層の除去が推奨されています。 以上の点を考慮し、材料変更に伴うリスクを評価し、適切な対策を講じることが重要です。 **使用したデータソース**: - 材料リマインダ(T096) - NKN006,"・一般的に言われているCr,Niの効果については概ね記載の通りかと思いますが、Crの浸炭促進効果については疑問です。 Crは酸化されやすく、浸炭中に表面にCr酸化膜を形成しCの侵入を阻害する傾向があります。 ・浸炭中、表層部ではCrは選択酸化されて粒界酸化という現象を起こします。粒界酸化の周辺では焼き入れ性の低下や、粒界酸化が切欠き効果となり強度を低下させる懸念もあります。また、Crは窒化物も形成しやすいのでNの焼き入れ性を阻害したり、CrNの粒界析出による強度低下も懸念されます。（焼入れ性についてはNi,Moも増加しているので影響は軽減されているかもしれません） ・Ni,Crの増加は残留オーステナイトの増加にもつながると考えられます。残留オーステナイトは使われ方によって挙動が変わると言われていますので一概に良し悪しの判断が難しいですが。 ・材料変更に伴う硬さ分布やミクロ組織の変化（狙い通り焼入れされているか、粒界酸化の深さは変化していないか、など）は抑えておく必要があると考えます。",Title:04-6_CZファスナーコート MajorCategory:各論 SubCategory:表面処理 Title:NAD116_応力計算の例6 MajorCategory:ノウハウ SubCategory:接着 Title:MKI00048_鋳鉄・鋳鋼の特性データ6 MajorCategory:各論 SubCategory:金属 Title:材料リマインダ(T008) MajorCategory:リマインダ SubCategory:シャシーC Title:材料リマインダ(S026) MajorCategory:リマインダ SubCategory:HV Title:材料リマインダ(E111) MajorCategory:リマインダ SubCategory:ボデーB Title:材料リマインダ(T096) MajorCategory:リマインダ SubCategory:エンジンE Title:未然防止留意点(MB007) MajorCategory:リマインダ SubCategory:未然防止留意点 Title:材料リマインダ(K099) MajorCategory:リマインダ SubCategory:ドラトレV Title:NKN006 MajorCategory:ノウハウ SubCategory:金属 Title:NKN006e MajorCategory:ノウハウ SubCategory:金属 Title:NKN009e MajorCategory:ノウハウ SubCategory:金属
2,SUS430（厚板）×YUS205-M1（薄板）をスポット溶接する仕様を検討中です。 従来はSUS430ではなくSUS430MTを使用していたが、SUS430材の使用ができなくなり上記組み合わせで鋭敏化の問題がないかのご確認いただきたいです。 （N=3、シュウ酸及び王水を使用） もしそもそもスポット溶接は鋭敏化対象外等もあるのであれば教えていただきたいです。 よろしくお願いいたします。,"### 回答 SUS430（厚板）とYUS205-M1（薄板）をスポット溶接する際の鋭敏化の問題について、以下の点を考慮する必要があります。 1. **鋭敏化のリスク**: SUS430は、一般的に鋭敏化を抑制する元素（TiやNb）が添加されていないため、鋭敏化が発生しやすい材料です。鋭敏化とは、ステンレス鋼が高温にさらされることでクロム炭化物が結晶粒界に生成し、その周囲がクロム欠乏状態になる現象です。これにより、粒界腐食が発生しやすくなります。 2. **スポット溶接の影響**: スポット溶接は短時間で高温に達するため、鋭敏化のリスクが高まります。特にSUS430のような鋭敏化しやすい材料を使用する場合、溶接部での鋭敏化が懸念されます。 3. **試験方法**: 鋭敏化の確認には、シュウ酸エッチング試験（TSG2353G）や王水エッチング試験が有効です。これらの試験により、溶接部の鋭敏化の有無を確認することができます。 4. **対策**: - **材料の選定**: SUS430の代わりに、鋭敏化を抑制する元素が添加されたSUS430MTやSUS436MTなどの材料を使用することが推奨されます。 - **溶接条件の最適化**: 溶接条件を最適化し、鋭敏化のリスクを最小限に抑えることが重要です。 5. **スポット溶接が鋭敏化の対象外かどうか**: スポット溶接は鋭敏化のリスクがあるため、対象外とは言えません。特にSUS430のような材料を使用する場合、鋭敏化のリスクを十分に考慮する必要があります。 ### リマインダ **材料リマインダ(K121)**: - SUS430は一般に、結晶粒の鋭敏化発生を抑制する元素（Ti, Nb）が添加されていないため、鋭敏化しやすい。 - SUS430を使用する場合、鋭敏化のリスクが高まるため、SUS436MTなどのTi添加鋼に変更することが推奨される。 **未然防止留意点(MB025)**: - ステンレス鋼の鋭敏化に関する留意事項として、溶接や熱処理で700°C程度の熱履歴を受けると鋭敏化が発生しやすい。 - SUS430などの鋼種は、鋭敏化を抑制するために低炭素材料やTi, Nb添加材料を使用することが推奨される。 **参考文献**: - 材料リマインダ(K121) - 未然防止留意点(MB025) これらの情報を基に、SUS430の使用について再検討し、適切な材料選定と溶接条件の最適化を行うことをお勧めします。","溶接部ではありませんが、一般部で縞状のエッチング模様、若干の鋭敏化がみられるようにみえます ###写真等のデータを踏まえた回答 溶接以前にCr炭化物が形成してしまっていると思われます ###写真等のデータを踏まえた回答 SUS430はSUS430MTと鋼種記号は似ていますが、異質な材料です。 SUS430はC量が高く、Ti,Nbの鋭敏化抑止の元素が入っていないため、 溶接や高温で使用(製造中含む)環境があり、腐食環境が考えられる部品では適した材料ではありません。",Title:材料リマインダ(K121) MajorCategory:リマインダ SubCategory:エンジンE

In [0]:
# Save the DataFrame as a table in the specified schema
from datetime import datetime,timedelta

# 現在の日時を取得
now = datetime.now()
# yyyyMMddHHmm形式の文字列に変換
new_time = now + timedelta(hours=9)
formatted_date = new_time.strftime('%Y%m%d_%H%M')

In [0]:
from datetime import datetime, timedelta

def save_df_to_catalog(df, table_name: str):
    # 現在のUTC日時を取得
    now_utc = datetime.utcnow()
    # 日本時間 (UTC+9) に変換
    now_japan = now_utc + timedelta(hours=9)
    # yyyyMMddHHmm形式の文字列に変換
    formatted_date_japan = now_japan.strftime('%Y%m%d_%H%M')

    df = spark.createDataFrame(df)

    # Create the schema in the specified catalog and schema
    spark.sql("CREATE SCHEMA IF NOT EXISTS hive_metastore.poc2")

    # Save the DataFrame as a table in the specified schema
    df.write.mode("overwrite").saveAsTable(f"poc2.{table_name}_{formatted_date_japan}")
    
    print(f"Table 'poc2.{table_name}_{formatted_date_japan}' was created.")
    check_df = spark.table(f"poc2.{table_name}_{formatted_date_japan}")
    display(check_df)

save_df_to_catalog(qa_results_df, 'extract_summary_results')

Table 'poc2.extract_summary_results_20240917_2006' was created.


質問ナンバー,質問,RAG回答,人間回答,検索されたドキュメント
1,チェーンの材質をSAE8620から18CrNiMo7-6に変更出来ないか検討しています。 SAE8620から18CrNiMo7-6への変更にあたっての心配点を洗い出す中でNiとCr添加の役割は下記と理解しています。 ・Niは強度、焼入れ性、靭性を向上する。 ・Cr : 焼入れ性、焼き戻し抵抗を向上＋浸炭促進効果がある。 故に、強度、耐摩耗性向上すると考えています。材料の一般的な知識がなく、上記の考え方が正しいのか、の確認をさせて頂きたいです。また、もし上記以外に材料成分が変わることによる背反、注意点があれば教えて頂けると幸いです。,### 回答 チェーンの材質をSAE8620から18CrNiMo7-6に変更する際の心配点と、NiとCr添加の役割について以下のように説明します。 #### NiとCr添加の役割 - **ニッケル (Ni)**: - **強度の向上**: Niは鋼の強度を向上させる効果があります。特に低温での靭性を高めるため、衝撃に対する耐性が向上します。 - **焼入れ性の向上**: Niは焼入れ性を向上させ、鋼の硬化深さを増加させます。これにより、部品全体が均一に硬化しやすくなります。 - **靭性の向上**: Niは靭性を向上させ、脆性破壊のリスクを低減します。 - **クロム (Cr)**: - **焼入れ性の向上**: Crは焼入れ性を向上させ、鋼の硬化深さを増加させます。これにより、部品全体が均一に硬化しやすくなります。 - **焼き戻し抵抗の向上**: Crは焼き戻し抵抗を向上させ、高温での強度を維持します。 - **浸炭促進効果**: Crは浸炭処理を促進し、表面硬度を高める効果があります。 #### 材料成分が変わることによる背反や注意点 - **熱処理の違い**: 18CrNiMo7-6はSAE8620よりも高い焼入れ性を持つため、熱処理条件を適切に設定する必要があります。過度な焼入れは内部応力を増加させ、割れやすくなる可能性があります。 - **コスト**: 18CrNiMo7-6は合金元素が多く含まれているため、材料コストが高くなる可能性があります。 - **加工性**: 高強度の材料は加工が難しくなることがあり、工具の摩耗が早くなる可能性があります。 - **摩耗特性**: 18CrNiMo7-6は高硬度を持つため、摩耗特性が向上しますが、相手材との摩耗関係も考慮する必要があります。 これらの点を考慮し、材料変更の際には適切な熱処理条件の設定やコスト評価、加工性の確認を行うことが重要です。 ### リマインダ - **材料リマインダ(T096)**: チェーンの摩耗による伸びに関するリマインダがあります。ピンとブッシュの摩耗がチェーンの伸びを引き起こす可能性があるため、表面処理や材料選定に注意が必要です。 **使用したデータソース**: - 材料リマインダ(T096) - NKN006,"・一般的に言われているCr,Niの効果については概ね記載の通りかと思いますが、Crの浸炭促進効果については疑問です。 Crは酸化されやすく、浸炭中に表面にCr酸化膜を形成しCの侵入を阻害する傾向があります。 ・浸炭中、表層部ではCrは選択酸化されて粒界酸化という現象を起こします。粒界酸化の周辺では焼き入れ性の低下や、粒界酸化が切欠き効果となり強度を低下させる懸念もあります。また、Crは窒化物も形成しやすいのでNの焼き入れ性を阻害したり、CrNの粒界析出による強度低下も懸念されます。（焼入れ性についてはNi,Moも増加しているので影響は軽減されているかもしれません） ・Ni,Crの増加は残留オーステナイトの増加にもつながると考えられます。残留オーステナイトは使われ方によって挙動が変わると言われていますので一概に良し悪しの判断が難しいですが。 ・材料変更に伴う硬さ分布やミクロ組織の変化（狙い通り焼入れされているか、粒界酸化の深さは変化していないか、など）は抑えておく必要があると考えます。",Title:04-6_CZファスナーコート MajorCategory:各論 SubCategory:表面処理 Title:NAD116_応力計算の例6 MajorCategory:ノウハウ SubCategory:接着 Title:MKI00048_鋳鉄・鋳鋼の特性データ6 MajorCategory:各論 SubCategory:金属 Title:材料リマインダ(T008) MajorCategory:リマインダ SubCategory:シャシーC Title:材料リマインダ(S026) MajorCategory:リマインダ SubCategory:HV Title:材料リマインダ(E111) MajorCategory:リマインダ SubCategory:ボデーB Title:材料リマインダ(T096) MajorCategory:リマインダ SubCategory:エンジンE Title:未然防止留意点(MB007) MajorCategory:リマインダ SubCategory:未然防止留意点 Title:材料リマインダ(K099) MajorCategory:リマインダ SubCategory:ドラトレV Title:NKN006 MajorCategory:ノウハウ SubCategory:金属 Title:NKN006e MajorCategory:ノウハウ SubCategory:金属 Title:NKN009e MajorCategory:ノウハウ SubCategory:金属
2,SUS430（厚板）×YUS205-M1（薄板）をスポット溶接する仕様を検討中です。 従来はSUS430ではなくSUS430MTを使用していたが、SUS430材の使用ができなくなり上記組み合わせで鋭敏化の問題がないかのご確認いただきたいです。 （N=3、シュウ酸及び王水を使用） もしそもそもスポット溶接は鋭敏化対象外等もあるのであれば教えていただきたいです。 よろしくお願いいたします。,### 回答 SUS430（厚板）とYUS205-M1（薄板）のスポット溶接における鋭敏化の問題について、以下の点を考慮する必要があります。 1. **SUS430の鋭敏化特性**: SUS430は、一般的に鋭敏化を抑制する元素（TiやNb）が添加されていないため、鋭敏化が発生しやすい材料です。鋭敏化とは、ステンレス鋼が高温にさらされることでクロム炭化物が結晶粒界に生成し、その周囲がクロム欠乏状態になる現象です。この状態になると、腐食環境にさらされると粒界腐食が発生しやすくなります。 2. **スポット溶接の影響**: スポット溶接は短時間で高温に達するため、鋭敏化のリスクがあります。特にSUS430のような鋭敏化しやすい材料では、溶接部での鋭敏化が懸念されます。 3. **試験方法**: 鋭敏化の確認には、シュウ酸エッチング試験（TSG2353G）や王水エッチング試験が有効です。これらの試験により、溶接部の鋭敏化の有無を確認することができます。 4. **対策**: - **材料の選定**: SUS430の代わりに、鋭敏化を抑制する元素が添加されたSUS430MTやSUS436MTなどの材料を使用することが推奨されます。 - **溶接条件の最適化**: 溶接条件を最適化し、鋭敏化を最小限に抑えるようにすることが重要です。 5. **スポット溶接が鋭敏化の対象外かどうか**: スポット溶接も鋭敏化の対象となります。特にSUS430のような材料では、溶接部での鋭敏化が発生する可能性が高いため、適切な試験と対策が必要です。 ### リマインダ **材料リマインダ(K121)**: - SUS430は、鋭敏化を抑制する元素（TiやNb）が添加されていないため、鋭敏化が発生しやすい。 - SUS430を使用する場合、鋭敏化のリスクがあるため、SUS436MTなどのTi添加鋼に変更することが推奨される。 **未然防止留意点(MB025)**: - ステンレス鋼の鋭敏化に関する留意事項として、溶接や熱処理で700°C程度の熱履歴を受けると鋭敏化が発生する可能性がある。 - SUS430などの鋼種は、鋭敏化を抑制するために低炭素材料やTi、Nb添加材料を使用することが推奨される。 **MKI00084_ステンレスの特性データ9**: - SUS430は高温に加熱されるとクロム炭化物が生成し、粒界腐食感受性が増すことがある。 - 鋭敏化防止のためには、低炭素材料やTi、Nb添加材料を使用することが有効である。 **参考文献**: - 材料リマインダ(K121) - 未然防止留意点(MB025) - MKI00084_ステンレスの特性データ9,"溶接部ではありませんが、一般部で縞状のエッチング模様、若干の鋭敏化がみられるようにみえます ###写真等のデータを踏まえた回答 溶接以前にCr炭化物が形成してしまっていると思われます ###写真等のデータを踏まえた回答 SUS430はSUS430MTと鋼種記号は似ていますが、異質な材料です。 SUS430はC量が高く、Ti,Nbの鋭敏化抑止の元素が入っていないため、 溶接や高温で使用(製造中含む)環境があり、腐食環境が考えられる部品では適した材料ではありません。",Title:材料リマインダ(K121) MajorCategory:リマインダ SubCategory:エ

In [0]:
# 
qa_results_df.to_excel(excel_path, sheet_name=f'{formatted_date}')